# Bloch vs PINN (Pointwise Picking Only)

This notebook compares:
1. Analytical Bloch branches (monoatomic + mass-in-mass),
2. PINN pointwise ridge picking (`extract_ridge`).

No continuity ridge tracking is used here.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    extract_ridge,
    linear_dispersion,
)
from dispersion_theory import monoatomic_dispersion, mass_in_mass_dispersion

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_file = '../Result/pinn_data.mat'

# Structural parameters for Bloch reference
k_coupling = 1.0
mx = 1.0
my = 0.3
k_int = 8.6483

omega_min = 0.01
skip_transient = 0.20
n_harmonic = 4
pts_per_cycle = 12

In [ ]:
# PINN spectrum + pointwise ridge
t_raw, x_raw, params = load_pinn_results(mat_file)
X_raw = x_raw.T

omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)
t_u, X_u, dt, omega_nyq = resample_to_uniform(
    t_raw, X_raw, omega_max_lin,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
)

k_pos, omega_pos, spectrum = compute_spectrum(t_u, X_u, skip_transient=skip_transient)
k_pw, omega_pw = extract_ridge(k_pos, omega_pos, spectrum, omega_min=omega_min)

# Bloch analytical curves
k_line = np.linspace(0, np.pi, 400)
omega_bloch_mono = monoatomic_dispersion(k_line, k_coupling, mx)
omega_bloch_lo, omega_bloch_hi = mass_in_mass_dispersion(k_line, k_coupling, mx, my, k_int)

print('Loaded:', mat_file)
print('params keys:', list(params.keys()))

In [ ]:
# Comparison plot: Bloch vs pointwise
plt.figure(figsize=(10, 6.5))

# Bloch reference
plt.plot(k_line/np.pi, omega_bloch_mono, 'k:', lw=1.8, label='Bloch monoatomic')
plt.plot(k_line/np.pi, omega_bloch_lo, 'k--', lw=2.0, label='Bloch acoustic (mass-in-mass)')
plt.plot(k_line/np.pi, omega_bloch_hi, 'k-.', lw=2.0, label='Bloch optical (mass-in-mass)')

# Pointwise ridge
valid_pw = np.isfinite(omega_pw)
plt.plot(k_pw[valid_pw]/np.pi, omega_pw[valid_pw], color='tab:orange', lw=2.4, label='PINN pointwise ridge')

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title('Bloch vs PINN pointwise picking')
plt.grid(alpha=0.25)
plt.legend(loc='best', fontsize=9, ncol=2)
plt.tight_layout()
plt.show()